In [14]:
import os
## unmasking, learned embeddings
# base_names = ['masking_clr_len200_3layers_ratio_30_lr001_updated',
#               'masking_clr_len200_3layers_ratio_50_lr001_updated',
#               'masking_clr_len200_3layers_ratio_70_lr001_updated',
#               'masking_clr_len200_7layers_ratio_30_lr001_updated',
#               'masking_clr_len200_7layers_ratio_50_lr001_updated',
#               'masking_clr_len200_7layers_ratio_70_updated',
#               'masking_log_rel_ab_len200_3layers_ratio_30_updated',
#               'masking_log_rel_ab_len200_3layers_ratio_50_updated',
#               'masking_log_rel_ab_len200_3layers_ratio_70_updated',
#               'masking_MSE_LOSS_log_rel_ab_len200_3layers_ratio_50_updated'
#               ]
## unmasking, evo2 fixed embeddings
base_names = ['masking_clr_len200_3layers_ratio_50_evo_mlp',
              'masking_clr_len200_3layers_ratio_70_evo_mlp',
              'masking_clr_len200_7layers_ratio_50_evo_mlp',
              'masking_clr_len200_7layers_ratio_70_evo_mlp',
              'masking_log_rel_ab_len200_3layers_ratio_50_evo_mlp',
              'masking_log_rel_ab_len200_3layers_ratio_70_evo_mlp',
              'masking_log_rel_ab_len200_7layers_ratio_50_evo_mlp',
              'masking_log_rel_ab_len200_7layers_ratio_70_evo_mlp'
             ]
tasks = ['location','age','bmi','sex','supplement','helicobacter_pylori_infection','hbv_infection']
emb_style = ['avg-pool', 'cls']
emb_name = {'avg-pool':'pool', 'cls':'cls'}
topdir = '/project/aip-rahulgk/dpellow/gut_microbiome_GPT/configs/finetune/evo/task_configs/'
os.makedirs(topdir, exist_ok=True)
list_file_name = os.path.join(topdir, 'config_list.txt')
with open(list_file_name, 'w') as list_file:
  for task in tasks:
    # make a directory if it doesn't exist
    task_dir = os.path.join(topdir, task)
    os.makedirs(task_dir, exist_ok=True)
    if task in ['age','bmi']:
      label = 'continuous_label'
      task_type = 'regression'
    else:
      label = 'categorical_label'
      task_type = 'classification'
    for n in base_names:
      for e in emb_style:
        norm = 'clr' if '_clr_' in n else 'log_rel_abundance' if '_log_rel_ab_' in n else None
        layers = 3 if '_3layers_' in n else 7 if '_7layers_' in n else None
    
    


        text = f'''
        # finetuning config - classification
        paths:
          downstream_train: "/project/aip-rahulgk/gutmodel/hmc_final_fixed/downstream_train.h5ad"
          downstream_test: "/project/aip-rahulgk/gutmodel/hmc_final_fixed/downstream_test.h5ad"
          output_dir: "/project/aip-rahulgk/dpellow/gut_microbiome_GPT/outputs/finetune/evo/{task}/{n}_{emb_name[e]}/"
          taxa_vocab_path: "/project/aip-rahulgk/dpellow/gut_microbiome_GPT/outputs/pretrain/evo/{n}/taxa_vocab.pkl"
          # batch_vocab_path: "./results/pretraining/batch_vocab.json"
          checkpoint_path: "/project/aip-rahulgk/dpellow/gut_microbiome_GPT/outputs/pretrain/evo/{n}/best_model/best_model/model.safetensors"
          model_config_path: "/project/aip-rahulgk/dpellow/gut_microbiome_GPT/outputs/finetune/evo/{task}/{n}_{emb_name[e]}/model_config.json"
        wandb:
          enabled: true
          entity: "haoze-deng-university-of-toronto"
          project: "microbiome-finetuning"
          run_name: {n}_{emb_name[e]}
          run_notes: "Finetuning on location classification task"

        data:
          norm_strategy: "{norm}"  # Should match pretraining
          split_key: null
          val_size: 0.2
          use_batch_labels: false
          max_seq_len: 200
          num_workers: 4
          downsample_distribution: null  # Not used in finetuning (no perturbation)
          
          # Finetuning-specific data settings
          label_column: "{label}"  # Column in adata.obs with labels
          finetune_task_name: "{task}"  # Value in adata.obs['downstream_task'] to filter


        training:
          seed: 423985693
          init_lr: 1e-4 
          batch_size: 64
          max_epochs: 50
          cosine_warmup_ratio_or_step: 100
          log_interval: 10
          patience: 10
          grad_accumulation_steps: 1
          enable_fp16: false
          notes: "Finetuning pretrained model on location classification"
          
          # Finetuning-specific settings
          finetune_mode: "full" 
          finetune_task: "{task_type}"  # choices: classification, regression
          tasks: []  # Empty for finetuning (not used, model uses finetune_forward)
          
          # Not used in finetuning but kept for compatibility
          train_mask_ratio: null
          contrastive_temperature: null
          masking_prob: null

        validation:
          batch_size: 64
          eval_interval_epochs: 1

        # Model (match pretrained)
        model:
          params:
            d_model: 128
            d_proj: 128
            seq_len: 200
            nhead: 8
            d_hid: 512
            nlayers: {layers}
            dropout: 0.1
            abundance_emb_style: "continuous"
            sample_emb_style: "{e}"
            use_gnn: false
            # For fixed taxa embeddings:
            preinitialized_taxa_embedding_path: "/project/aip-rahulgk/gutmodel/hmc_final_fixed/qwen3_taxa_embeddings.npy"
            freeze_preinitialized_embeddings: true # frozen during pretraining
            preinitialized_embedding_projection: "mlp" # choices: mlp, linear


        debug:
          # nrows: null
          start_over: false
        '''
        fname = f'{n}_{emb_name[e]}.yaml'
        fpath = os.path.join(task_dir, fname)
        with open(fpath, 'w') as f:
          f.write(text)
        list_file.write(fpath + '\n')
      # print(80*"#")
      # print(text)
      

In [10]:
!pwd

/project/6101781/dpellow/gut_microbiome_GPT/notebooks
